# Wazire Execution Agent - LOXM Style Trading Bot

This notebook implements a reinforcement learning-based execution agent similar to JPMC's LOXM.

## Architecture Overview
- **Framework**: Deep Q-Network (DQN) using PyTorch
- **Environment**: Custom Gymnasium environment for trading execution
- **Goal**: Minimize implementation shortfall while managing market impact

# MVP Meetup Notes — Online Exam System
Good—focusing on an MVP is exactly the right move. If you try to build the full proctoring + national system immediately, it will stall. The MVP should **solve the core pain: missing scripts + slow marking + poor result tracking**.

---

# 🎯 MVP GOAL (Keep This Tight)

> Build a **reliable exam + marking + result system for ONE school**

Ignore for now:

* AI proctoring
* Camera/mic monitoring
* National scaling

---

# 🧱 MVP FEATURES (ONLY WHAT MATTERS)

## **1. Authentication + Roles**

Users:

* Student
* Lecturer
* Admin

👉 Must have:

* Login / signup
* Role-based dashboards

---

## **2. Course & Exam Setup**

Lecturer/Admin can:

* Create course (e.g., CSC101)
* Create exam:

  * Title
  * Duration
  * Start/end time

---

## **3. Question System**

Support ONLY:

* MCQ (start simple)
* (Optional later: essay)

👉 Features:

* Add questions
* Add options
* Set correct answer

---

## **4. Exam Taking (CORE)**

Students can:

* Start exam
* Answer questions
* Submit

👉 MUST HAVE:

* **Auto-save answers (very important)**
* Timer
* One attempt

---

## **5. Auto Marking**

* MCQs marked instantly
* Score calculated automatically

---

## **6. Results System (MAIN VALUE)**

* Students see results immediately
* Lecturers see all student scores
* No missing scripts (everything stored)

---

## **7. Basic Admin Panel**

Admin can:

* View users
* View exams
* View results

---

# 🚫 DO NOT BUILD YET

These will slow you down:

* Webcam monitoring
* AI cheating detection
* Mobile app
* Complex analytics
* Multi-school support

---

# 🏗️ TECH STACK (Keep It Simple)

## **Frontend**

* React

## **Backend**

* Node.js (Express)

## **Database**

* PostgreSQL
  OR
* Firebase (faster MVP)

---

# 🗄️ SIMPLE DATABASE DESIGN

## Users

* id
* name
* email
* role

## Courses

* id
* name
* lecturer_id

## Exams

* id
* course_id
* title
* duration

## Questions

* id
* exam_id
* question
* options
* correct_answer

## Submissions

* id
* student_id
* exam_id
* answers
* score

---

# ⚙️ MVP FLOW (IMPORTANT)

### Lecturer:

1. Create course
2. Create exam
3. Add questions

### Student:

1. Login
2. Start exam
3. Answer questions
4. Submit

### System:

* Auto-marks
* Stores result
* Displays instantly

---

# ⏱️ 30-DAY BUILD PLAN

## **Week 1**

* Auth system (login/register)
* Role system
* Basic UI

## **Week 2**

* Course + exam creation
* Question system (MCQ)

## **Week 3**

* Exam-taking interface
* Timer + autosave

## **Week 4**

* Auto-marking
* Results dashboard
* Testing + fixes

---

# 🔑 WHAT MAKES YOUR MVP STRONG

Focus on these 3 things:

### ✅ 1. Reliability

* Auto-save answers every few seconds
* Prevent data loss

### ✅ 2. Simplicity

* Clean UI
* Easy for lecturers

### ✅ 3. Speed

* Instant results

---

# 🚀 AFTER MVP (NEXT STEPS)

Only AFTER MVP works:

1. Add essay marking
2. Add analytics
3. Add anti-cheating features
4. Expand to multiple schools
5. Pitch to institutions

---

# 🎯 FINAL ANSWER

> Your MVP should be a simple, reliable online exam system with role-based access, MCQ exams, auto-marking, and instant results. Focus on eliminating missing scripts and speeding up marking before adding complex features like AI proctoring or multi-institution support.


## 1. Setup and Dependencies

In [1]:
# Install required packages
!pip install torch gymnasium stable-baselines3 numpy pandas matplotlib yfinance

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.1.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached multitasking-0.0.12.tar.gz (19 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached websockets-16.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 17.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 23.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 23.8 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 2

In [2]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
import yfinance as yf
from typing import Tuple, Dict, Any

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0
CUDA available: False


## 2. Custom Trading Environment

In [3]:
class ExecutionEnvironment(gym.Env):
    """Custom trading environment for execution optimization."""
    
    def __init__(self, total_shares: int = 10000, time_horizon: int = 390, symbol: str = 'AAPL'):
        super().__init__()
        
        # Action space: 0=Wait, 1=Passive, 2=Neutral, 3=Aggressive
        self.action_space = gym.spaces.Discrete(4)
        
        # Observation space: [inventory_ratio, time_ratio, spread, volatility, shortfall]
        self.observation_space = gym.spaces.Box(
            low=0, high=1, shape=(5,), dtype=np.float32
        )
        
        self.total_shares = total_shares
        self.initial_shares = total_shares
        self.time_horizon = time_horizon
        self.time_left = time_horizon
        self.symbol = symbol
        
        # Market data simulation
        self._load_market_data()
        self.current_step = 0
        self.arrival_price = 0
        
    def _load_market_data(self):
        """Load historical market data for simulation."""
        # Download 1 year of daily data
        ticker = yf.Ticker(self.symbol)
        data = ticker.history(period='1y', interval='1d')
        
        # Calculate features
        data['returns'] = data['Close'].pct_change()
        data['volatility'] = data['returns'].rolling(20).std()
        data['spread'] = (data['High'] - data['Low']) / data['Close']
        
        self.market_data = data.dropna().reset_index(drop=True)
        
    def reset(self, seed=None, options=None) -> Tuple[np.ndarray, Dict]:
        """Reset environment to initial state."""
        super().reset(seed=seed)
        
        self.total_shares = self.initial_shares
        self.time_left = self.time_horizon
        self.current_step = np.random.randint(0, len(self.market_data) - self.time_horizon)
        
        # Set arrival price
        self.arrival_price = self.market_data.iloc[self.current_step]['Close']
        
        return self._get_obs(), {}
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        """Execute one time step."""
        # Get current market data
        current_data = self.market_data.iloc[self.current_step]
        current_price = current_data['Close']
        
        # Calculate execution size based on action
        action_sizes = [0.0, 0.01, 0.05, 0.15]  # Percentage of remaining shares
        execution_size = int(self.total_shares * action_sizes[action])
        
        # Simulate market impact
        market_impact = action * 0.001  # Larger actions have more impact
        execution_price = current_price * (1 - market_impact)
        
        # Calculate reward (negative implementation shortfall)
        shortfall = (self.arrival_price - execution_price) * execution_size
        time_penalty = 0.1 * (self.total_shares / self.initial_shares) * (self.time_horizon - self.time_left) / self.time_horizon
        reward = -shortfall - time_penalty * 1000  # Scale reward
        
        # Update state
        self.total_shares -= execution_size
        self.time_left -= 1
        self.current_step += 1
        
        # Check if episode is done
        done = self.total_shares <= 0 or self.time_left <= 0
        truncated = False
        
        return self._get_obs(), reward, done, truncated, {}
    
    def _get_obs(self) -> np.ndarray:
        """Get current observation."""
        current_data = self.market_data.iloc[self.current_step]
        
        inventory_ratio = self.total_shares / self.initial_shares
        time_ratio = self.time_left / self.time_horizon
        spread = current_data['spread']
        volatility = current_data['volatility']
        shortfall = (self.arrival_price - current_data['Close']) / self.arrival_price
        
        # Normalize features to [0, 1]
        obs = np.array([
            inventory_ratio,
            time_ratio,
            min(spread * 100, 1.0),  # Scale and cap spread
            min(volatility * 50, 1.0),  # Scale and cap volatility
            max(0, min(abs(shortfall) * 10, 1.0))  # Scale and cap shortfall
        ], dtype=np.float32)
        
        return obs

## 3. Test the Environment

In [4]:
# Create and test the environment
env = ExecutionEnvironment(total_shares=10000, time_horizon=100, symbol='AAPL')

# Test reset
obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Initial observation: {obs}")

# Test a few steps
for i in range(5):
    action = env.action_space.sample()  # Random action
    obs, reward, done, truncated, info = env.step(action)
    print(f"Step {i+1}: Action={action}, Reward={reward:.4f}, Done={done}")
    if done:
        break

Initial observation shape: (5,)
Initial observation: [1. 1. 1. 1. 0.]
Step 1: Action=2, Reward=-195.3984, Done=False
Step 2: Action=2, Reward=398.6927, Done=False
Step 3: Action=0, Reward=-1.8050, Done=False
Step 4: Action=2, Reward=6460.9433, Done=False
Step 5: Action=1, Reward=1413.8040, Done=False


## 4. Define the DQN Model

In [5]:
# Create the DQN model
model = DQN(
    'MlpPolicy',
    env,
    learning_rate=1e-4,
    buffer_size=10000,
    learning_starts=1000,
    batch_size=32,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    replay_buffer_class=None,
    replay_buffer_kwargs=None,
    optimize_memory_usage=False,
    target_update_interval=1000,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    max_grad_norm=10,
    verbose=1,
    tensorboard_log=None,
    policy_kwargs=dict(
        net_arch=[64, 64],
        activation_fn=nn.ReLU
    )
)

print("DQN model created successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.policy.parameters())}")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DQN model created successfully!
Model parameters: 9608


## 5. Train the Model

In [7]:
# Train the model
print("Starting training...")
model.learn(
    total_timesteps=50000,
    log_interval=1000,
    progress_bar=True
)
print("Training completed!")

Starting training...


ImportError: You must install tqdm and rich in order to use the progress bar callback. It is included if you install stable-baselines with the extra packages: `pip install stable-baselines3[extra]`

## 6. Evaluate the Trained Agent

In [ ]:
def evaluate_agent(model, env, n_episodes=10):
    """Evaluate the trained agent."""
    episode_rewards = []
    execution_costs = []
    
    for episode in range(n_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            episode_reward += reward
        
        episode_rewards.append(episode_reward)
        
        # Calculate execution cost
        final_price = env.market_data.iloc[env.current_step - 1]['Close']
        execution_cost = (env.arrival_price - final_price) / env.arrival_price
        execution_costs.append(execution_cost)
        
        print(f"Episode {episode + 1}: Reward = {episode_reward:.2f}, Execution Cost = {execution_cost:.4f}")
    
    return episode_rewards, execution_costs

# Evaluate the agent
rewards, costs = evaluate_agent(model, env, n_episodes=10)

print(f"\nAverage Reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")
print(f"Average Execution Cost: {np.mean(costs):.4f} ± {np.std(costs):.4f}")

## 7. Visualize Results

In [ ]:
# Plot evaluation results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Episode rewards
ax1.plot(rewards, 'b-o', linewidth=2, markersize=6)
ax1.set_title('Episode Rewards')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.grid(True, alpha=0.3)

# Execution costs
ax2.plot(costs, 'r-o', linewidth=2, markersize=6)
ax2.set_title('Execution Costs')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Execution Cost (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Save the Model

In [ ]:
# Save the trained model
model.save("wazire_execution_agent")
print("Model saved as 'wazire_execution_agent.zip'")

# Save environment parameters for later use
import json
env_params = {
    'total_shares': env.initial_shares,
    'time_horizon': env.time_horizon,
    'symbol': env.symbol
}

with open('env_params.json', 'w') as f:
    json.dump(env_params, f)

print("Environment parameters saved to 'env_params.json'")

## 9. Load and Test Saved Model

In [ ]:
# Load the saved model
loaded_model = DQN.load("wazire_execution_agent")
print("Model loaded successfully!")

# Test the loaded model
obs, info = env.reset()
done = False
step_count = 0

print("\nTesting loaded model:")
while not done and step_count < 50:  # Limit to 50 steps for demo
    action, _states = loaded_model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    
    action_names = ['Wait', 'Passive', 'Neutral', 'Aggressive']
    print(f"Step {step_count + 1}: {action_names[action]} -> Reward: {reward:.4f}, Shares left: {env.total_shares}")
    
    step_count += 1
    
    if done:
        break

## 10. Next Steps and Production Considerations

### Phased Deployment Strategy:
1. **Phase 1 - Backtesting**: Train on extensive historical data
2. **Phase 2 - Paper Trading**: Test with live market data without real money
3. **Phase 3 - Guardrail Implementation**: Add safety layers before live trading

### Safety Measures to Implement:
- Position size limits
- Maximum drawdown controls
- Real-time monitoring dashboards
- Emergency stop mechanisms
- Compliance checks

### Model Improvements:
- Add more market features (order book depth, volume profiles)
- Implement more sophisticated reward functions
- Use ensemble methods for robustness
- Add regime detection for different market conditions